In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import split, col, trim


def parse_group_label_columns(df: DataFrame):
    """
    Parse and normalize group metadata from a pipe-separated group_label column.

    The input DataFrame is expected to contain a column `group_label` where
    multiple hierarchical attributes are encoded as a pipe-separated string.

    Example input:
        "South | Region A | City X | Segment Y | Product Z | Bucket 1"

    The function splits this column into separate, trimmed columns:
        - macro_region
        - region
        - municipality
        - segment
        - product_type
        - consumption_bucket

    Processing steps:
        - Splits `group_label` by pipe ("|")
        - Extracts each component into its own column
        - Trims whitespace from each value
        - Drops the original `group_label` and intermediate column

    Args:
        df: Spark DataFrame containing a `group_label` column.

    Returns:
        A Spark DataFrame with normalized group metadata columns.

    Side Effects:
        None (pure transformation).
    """
    df_split = df.withColumn(
        "parts",
        split(col("group_label"), "\\|")
    )

    df_parsed = (
        df_split
        .withColumn("macro_region", trim(col("parts")[0]))
        .withColumn("region", trim(col("parts")[1]))
        .withColumn("municipality", trim(col("parts")[2]))
        .withColumn("segment", trim(col("parts")[3]))
        .withColumn("product_type", trim(col("parts")[4]))
        .withColumn("consumption_bucket", trim(col("parts")[5]))
        .drop("group_label", "parts")
    )

    return df_parsed


def write_group_data_to_silver(df: DataFrame, destination: str):
    """
    Write cleaned group metadata to a silver-layer table.

    Args:
        df: Spark DataFrame containing parsed group metadata.
        destination: Fully qualified destination table name.

    Returns:
        None.

    Side Effects:
        - Overwrites the destination table.
        - Writes data to the metastore.
        - Prints a success message.
    """
    df.write \
        .mode("overwrite") \
        .saveAsTable(destination)

    print(f"Successfully wrote group data to {destination}")


if __name__ == "__main__":

    catalog = "fortum_challenge_data"
    schema = "01_bronze"
    table = "groups_data_raw"

    source = f"{catalog}.{schema}.{table}"
    destination = "fortum_challenge_data.02_silver.group_data_clean"

    df_raw = spark.table(source)

    df_clean = parse_group_label_columns(df_raw)

    write_group_data_to_silver(df_clean, destination)

    df_clean.display()